In [ ]:
import subprocess, sys
# 用 conda 装，不用 pip
subprocess.run(["conda", "run", "-n", "mlenv", "conda", "install", 
                "-c", "conda-forge", "faiss-cpu", "-y"], check=True)



In [4]:
import faiss
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

print("faiss:", faiss.__version__)
print("torch:", torch.__version__)
print("numpy:", np.__version__)
print("准备好，继续加载数据和模型")

faiss: 1.10.0
torch: 2.11.0
numpy: 2.4.3
准备好，继续加载数据和模型


In [5]:
# CELL 2
# ── 加载原始数据 ─────────────────────────────────────────────
ratings = pd.read_csv("../data/ml-1m/ratings.dat", sep="::",
    names=["user_id","movie_id","rating","timestamp"], engine="python")
movies = pd.read_csv("../data/ml-1m/movies.dat", sep="::",
    names=["movie_id","title","genres"], engine="python", encoding="latin-1")
users = pd.read_csv("../data/ml-1m/users.dat", sep="::",
    names=["user_id","gender","age","occupation","zip"], engine="python")

# ── 唯一 index 体系 ──────────────────────────────────────────
all_users  = sorted(ratings["user_id"].unique())
all_movies = sorted(ratings["movie_id"].unique())
user2idx   = {u: i for i, u in enumerate(all_users)}
movie2idx  = {m: i for i, m in enumerate(all_movies)}
idx2movie  = {i: m for m, i in movie2idx.items()}

# ── 修正后的模型定义 ─────────────────────────────────────────
class TwoTowerV2(nn.Module):
    def __init__(self, n_users, n_movies, emb_dim=32, tower_dims=[64, 32], n_occ=21):
        super().__init__()
        self.user_emb  = nn.Embedding(n_users, emb_dim)
        self.occ_emb   = nn.Embedding(n_occ, 16)
        self.movie_emb = nn.Embedding(n_movies, emb_dim)

        self.user_tower = nn.Sequential(
            nn.Linear(emb_dim + 16 + 1 + 1, tower_dims[0]), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(tower_dims[0], tower_dims[1])
        )
        self.movie_tower = nn.Sequential(
            nn.Linear(emb_dim + 18, tower_dims[0]), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(tower_dims[0], tower_dims[1])
        )
        # 三个 buffer，从 checkpoint 恢复，不需要外部传入
        self.register_buffer("genre_matrix", torch.zeros(n_movies, 18))
        self.register_buffer("age_min",   torch.tensor(0.0))
        self.register_buffer("age_range", torch.tensor(1.0))

    def get_item_vec(self, movie_idx):
        emb   = self.movie_emb(movie_idx)
        genre = self.genre_matrix[movie_idx]
        return self.movie_tower(torch.cat([emb, genre], dim=-1))

    def get_user_vec(self, user_idx, gender, age, occ):
        age_norm = (age - self.age_min) / self.age_range
        emb      = self.user_emb(user_idx)
        o_emb    = self.occ_emb(occ)
        return self.user_tower(torch.cat([emb, o_emb, gender, age_norm], dim=-1))

    def forward(self, user_idx, movie_idx, gender, age, occ):
        u_vec = self.get_user_vec(user_idx, gender, age, occ)
        i_vec = self.get_item_vec(movie_idx)
        return torch.sigmoid((u_vec * i_vec).sum(dim=-1))

# ── 加载权重 ─────────────────────────────────────────────────
# device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
device = "cpu"
ckpt   = torch.load("../data/two_tower_v2_best.pth", map_location="cpu")
hp     = ckpt["hparams"]

model = TwoTowerV2(n_users=hp["n_users"], n_movies=hp["n_movies"],
                   emb_dim=hp["emb_dim"], tower_dims=hp["tower_dims"])
model.load_state_dict(ckpt["model_state"])
model.to(device)
model.eval()

print(f"device: {device}")
print(f"age_min: {model.age_min.item()},  age_range: {model.age_range.item()}")
print(f"genre_matrix shape: {model.genre_matrix.shape}")
print(f"参数量: {sum(p.numel() for p in model.parameters()):,}")

device: cpu
age_min: 1.0,  age_range: 55.0
genre_matrix shape: torch.Size([3706, 18])
参数量: 322,896


In [6]:
# CELL 3
# ── 离线计算所有 item tower 向量 ─────────────────────────────
all_movie_idx = torch.arange(len(movie2idx), device=device)  # [3706]

with torch.no_grad():
    item_vecs = model.get_item_vec(all_movie_idx)             # [3706, 32]

item_vecs_np = item_vecs.cpu().numpy().astype("float32")      # faiss 要求 float32

# ── 建 Faiss 索引（内积，配合后面 L2 归一化 = 余弦相似度）────
dim = item_vecs_np.shape[1]                                   # 32
faiss.normalize_L2(item_vecs_np)                              # 原地 L2 归一化

index = faiss.IndexFlatIP(dim)                                # IP = Inner Product
index.add(item_vecs_np)                                       # 加入所有向量

# ── 验证 ─────────────────────────────────────────────────────
print(f"item_vecs shape: {item_vecs_np.shape}")               # (3706, 32)
print(f"Faiss index 中向量数: {index.ntotal}")                # 3706
print(f"向量 L2 范数（前3个，归一化后应为1.0）: "
      f"{np.linalg.norm(item_vecs_np[:3], axis=1).round(4)}")

item_vecs shape: (3706, 32)
Faiss index 中向量数: 3706
向量 L2 范数（前3个，归一化后应为1.0）: [1. 1. 1.]


In [ ]:
#CELL 3.1
batch_size = 256
all_vecs = []

with torch.no_grad():
    for start in range(0, len(movie2idx), batch_size):
        end     = min(start + batch_size, len(movie2idx))
        idx_    = torch.arange(start, end)
        vecs    = model.get_item_vec(idx_)
        all_vecs.append(vecs.numpy().astype("float32"))
        
item_vecs_np = np.vstack(all_vecs)
faiss.normalize_L2(item_vecs_np)

dim   = item_vecs_np.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(item_vecs_np)

print(f"item_vecs shape: {item_vecs_np.shape}")
print(f"Faiss index 中向量数: {index.ntotal}")
print(f"范数（前3个）: {np.linalg.norm(item_vecs_np[:3], axis=1).round(4)}")

In [7]:
QUERY_USER_ID = 1
u = users[users["user_id"] == QUERY_USER_ID].iloc[0]

gender_val = 0.0 if u["gender"] == "F" else 1.0
age_val    = float(u["age"])
occ_val    = int(u["occupation"])

print(f"查询用户：id={QUERY_USER_ID}, gender={u['gender']}, "
      f"age={u['age']}, occupation={occ_val}")

with torch.no_grad():
    u_vec = model.get_user_vec(
        user_idx = torch.tensor([user2idx[QUERY_USER_ID]]),
        gender   = torch.tensor([[gender_val]]),
        age      = torch.tensor([[age_val]]),
        occ      = torch.tensor([occ_val]),
    )

u_vec_np = u_vec.numpy().astype("float32")
faiss.normalize_L2(u_vec_np)

K = 20
scores, indices = index.search(u_vec_np, K)

watched = set(ratings[ratings["user_id"] == QUERY_USER_ID]["movie_id"].tolist())
print(f"该用户看过 {len(watched)} 部电影\n")

print(f"{'排名':<4} {'电影':<50} {'分数':>6}  {'已看'}")
print("-" * 70)
for rank, (idx_, score) in enumerate(zip(indices[0], scores[0]), 1):
    movie_id = idx2movie[idx_]
    title    = movies[movies["movie_id"] == movie_id]["title"].values[0]
    seen     = "✓" if movie_id in watched else ""
    print(f"{rank:<4} {title:<50} {score:.4f}  {seen}")

查询用户：id=1, gender=F, age=1, occupation=10
该用户看过 53 部电影

排名   电影                                                     分数  已看
----------------------------------------------------------------------
1    Bug's Life, A (1998)                               0.4954  ✓
2    Shakespeare in Love (1998)                         0.4878  
3    Toy Story (1995)                                   0.4387  ✓
4    Galaxy Quest (1999)                                0.4319  
5    Toy Story 2 (1999)                                 0.4275  ✓
6    American Beauty (1999)                             0.4247  
7    Babe (1995)                                        0.4077  
8    Austin Powers: The Spy Who Shagged Me (1999)       0.3954  
9    Edward Scissorhands (1990)                         0.3869  
10   Forrest Gump (1994)                                0.3839  
11   Groundhog Day (1993)                               0.3781  
12   Sixth Sense, The (1999)                            0.3743  ✓
13   Aladdin (1992)   